# Phase 1: Model B — encoder–decoder language model


## Imports and device


In [1]:
from pathlib import Path
import contextlib
import gc
import hashlib
import json
import math
import os
import random
import re
import shutil
import time
import zipfile

import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download, HfApi

try:
    from kaggle_secrets import UserSecretsClient
except Exception:
    UserSecretsClient = None

SEED = 42
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(i, torch.cuda.get_device_name(i), round(props.total_memory / 2**30, 2), "GiB")

if torch.cuda.is_available() and torch.cuda.device_count() < 2:
    print("WARNING: training is configured for T4 x2; select the 2-GPU accelerator before starting production training.")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU count: 2
0 Tesla T4 14.56 GiB
1 Tesla T4 14.56 GiB


## Configuration


In [2]:
OUTPUT_DIR = Path("/kaggle/working/cs781-phase1-encdec-50m")
RUN_NAME = "encdec_5e5d_640_50m_2gpu"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_LENGTH = 1536
TARGET_LENGTH = 512
RAW_BLOCK_LENGTH = SOURCE_LENGTH + TARGET_LENGTH

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_DATASET_REVISION = None  # latest official revision for the test file
TEST_FILENAME = "Phase_1_Pretraining/Test/test.json"

VERIFY_TOKEN_HASHES = False  # manifest + file sizes are still checked; full 200M stream SHA verification is slow
AUTO_RESUME = True
AUTO_MICROBATCH = True
SESSION_STOP_STEP = None     # e.g. 500 for an orderly Kaggle-session stop; None = full 50M run
RUN_FULL_TRAINING = True # set True only after all tests pass
RUN_SUBMISSION = True     # set True after training completes

HF_TOKEN = None
if UserSecretsClient is not None:
    try:
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

print("Output:", OUTPUT_DIR)
print("Run:", RUN_NAME)

Output: /kaggle/working/cs781-phase1-encdec-50m
Run: encdec_5e5d_640_50m_2gpu


## Packed token store


In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

TOKEN_STORE_REPO = "I-230482/cs781-phase1-tokenstore"

# Read token from Kaggle Secrets
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN is missing from Kaggle Secrets"

api = HfApi(token=HF_TOKEN)

# ------------------------------------------------------------
# 1. Check whether the token itself is valid
# ------------------------------------------------------------

me = api.whoami()

print("Authenticated as:", me["name"])

# ------------------------------------------------------------
# 2. Check whether THIS dataset repo is accessible
# ------------------------------------------------------------

info = api.repo_info(
    repo_id=TOKEN_STORE_REPO,
    repo_type="dataset",
)

print("Repo:", info.id)
print("Revision:", info.sha)
print("Private:", info.private)
print("AUTH + REPO ACCESS PASSED ✅")

Authenticated as: I-230482
Repo: I-230482/cs781-phase1-tokenstore
Revision: 93118e2c5f79366fa81e19b6ffaca7478214c1d9
Private: True
AUTH + REPO ACCESS PASSED ✅


In [4]:
from pathlib import Path
from huggingface_hub import snapshot_download

TOKEN_STORE_REPO = "I-230482/cs781-phase1-tokenstore"

TOKEN_STORE = Path(
    snapshot_download(
        repo_id=TOKEN_STORE_REPO,
        repo_type="dataset",
        token=HF_TOKEN,
        local_dir="/kaggle/working/tokens_phase1_200m_fast_v2",
    )
)

print("Downloaded token store:")
print(TOKEN_STORE)

manifest_path = (
    TOKEN_STORE
    / "token_manifest.json"
)

assert manifest_path.is_file(), (
    f"token_manifest.json not found in {TOKEN_STORE}"
)

print("TOKEN STORE READY ✅")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Downloaded token store:
/kaggle/working/tokens_phase1_200m_fast_v2
TOKEN STORE READY ✅


In [5]:
import json

TOKEN_MANIFEST = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)

train_tokens = sum(
    int(s["tokens"])
    for s in TOKEN_MANIFEST["streams"]
    if s["split"] == "train"
)

val_tokens = sum(
    int(s["tokens"])
    for s in TOKEN_MANIFEST["streams"]
    if s["split"] == "validation"
)

print("Available train tokens:", f"{train_tokens:,}")
print("Available validation tokens:", f"{val_tokens:,}")

assert train_tokens >= 50_000_000
assert val_tokens >= 1_000_000

print("MANIFEST READY ✅")

Available train tokens: 200,097,711
Available validation tokens: 5,038,406
MANIFEST READY ✅


In [6]:
def _json_hash(obj):
    return hashlib.sha256(
        json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    ).hexdigest()


def _read_manifest(path):
    obj = json.loads(Path(path).read_text(encoding="utf-8"))
    check = dict(obj)
    checksum = check.pop("manifest_hash", None)
    if checksum is None or _json_hash(check) != checksum:
        raise ValueError("Manifest checksum mismatch")
    return obj


def discover_token_store():
    preferred = [
        Path("/kaggle/working/legal-lm-phase1-optimized/tokens_phase1_200m_fast_v2"),
        Path("/kaggle/working/tokens_phase1_200m_fast_v2"),
    ]
    candidates = []

    seen = set()
    manifest_paths = []
    for p in preferred:
        m = p / "token_manifest.json"
        if m.is_file():
            manifest_paths.append(m)

    for root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if root.is_dir():
            manifest_paths.extend(root.rglob("token_manifest.json"))

    for manifest_path in manifest_paths:
        manifest_path = manifest_path.resolve()
        if manifest_path in seen:
            continue
        seen.add(manifest_path)
        try:
            m = _read_manifest(manifest_path)
            train_tokens = sum(int(s["tokens"]) for s in m["streams"] if s["split"] == "train")
            val_tokens = sum(int(s["tokens"]) for s in m["streams"] if s["split"] == "validation")
            if int(m.get("vocab_size", -1)) == 32000 and train_tokens >= 50_000_000 and val_tokens >= 1_000_000:
                candidates.append((manifest_path.parent, m, train_tokens, val_tokens))
        except Exception:
            pass

    if not candidates:
        raise FileNotFoundError(
            "Could not find a compatible Phase-1 packed token store. Mount/restore tokens_phase1_200m_fast_v2 first."
        )

    candidates.sort(key=lambda x: ("tokens_phase1_200m_fast_v2" in str(x[0]), x[2]), reverse=True)
    print("Compatible token stores:")
    for p, _, tr, va in candidates:
        print(" ", p, "train=", tr, "validation=", va)
    return candidates[0][0], candidates[0][1]


TOKEN_STORE, TOKEN_MANIFEST = discover_token_store()
print()
print("Using token store:", TOKEN_STORE)
print("Manifest hash:", TOKEN_MANIFEST["manifest_hash"])
print("Available train tokens:", sum(int(s["tokens"]) for s in TOKEN_MANIFEST["streams"] if s["split"] == "train"))

Compatible token stores:
  /kaggle/working/tokens_phase1_200m_fast_v2 train= 200097711 validation= 5038406

Using token store: /kaggle/working/tokens_phase1_200m_fast_v2
Manifest hash: a9eaf6d0c2c1d99d5f83ce8506fdc08cd729908556eae796217f7c8e8c0e638f
Available train tokens: 200097711


## Original tokenizer


In [7]:
def file_sha256(path, block=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()


def tokenizer_identity(path):
    path = Path(path)
    names = ("tokenizer.json", "tokenizer_config.json", "special_tokens_map.json")
    hashes = {name: file_sha256(path / name) for name in names if (path / name).is_file()}
    if "tokenizer.json" not in hashes:
        raise FileNotFoundError(f"Missing tokenizer.json in {path}")
    return {"files": hashes, "sha256": _json_hash(hashes)}


def discover_tokenizer(manifest):
    expected = manifest.get("identity", {}).get("tokenizer", {}).get("sha256")
    files = []
    for root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if root.is_dir():
            files.extend(root.rglob("tokenizer.json"))

    matches = []
    for f in files:
        d = f.parent
        try:
            ident = tokenizer_identity(d)
            tok = AutoTokenizer.from_pretrained(str(d), local_files_only=True, use_fast=True)
            special = [tok.pad_token_id, tok.unk_token_id, tok.bos_token_id, tok.eos_token_id]
            if tok.is_fast and len(tok) == 32000 and special == [0, 1, 2, 3]:
                if expected is None or ident["sha256"] == expected:
                    matches.append((d, tok, ident))
        except Exception:
            pass

    if not matches:
        raise FileNotFoundError(
            "Could not find the exact 32k tokenizer used by the packed token store. Mount the tokenizer dataset."
        )
    return matches[0]


TOKENIZER_PATH, tokenizer, TOKENIZER_IDENTITY = discover_tokenizer(TOKEN_MANIFEST)
TOKENIZER_HASH = TOKENIZER_IDENTITY["sha256"]
VOCAB_SIZE = len(tokenizer)

print("Tokenizer:", TOKENIZER_PATH)
print("Vocab:", VOCAB_SIZE)
print("Special IDs:", [tokenizer.pad_token_id, tokenizer.unk_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id])
print("Tokenizer hash:", TOKENIZER_HASH)

Tokenizer: /kaggle/input/datasets/jaskaran7d6/tokenizer
Vocab: 32000
Special IDs: [0, 1, 2, 3]
Tokenizer hash: 3163063597ba6358484fc37200c76f548040a2a58186862b519904ba427deb2b


## Dataset module


In [8]:
%%writefile legal_encdec_data.py
from __future__ import annotations
import bisect, hashlib, json, math, random
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, Sampler


def require(c,m):
    if not c: raise ValueError(m)

def json_hash(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False,separators=(",",":")).encode()).hexdigest()

def file_hash(path, block_size=8*1024*1024):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(block_size),b''): h.update(b)
    return h.hexdigest()

def read_json(path): return json.loads(Path(path).read_text(encoding='utf-8'))

def tokenizer_identity(path):
    path=Path(path)
    names=("tokenizer.json","tokenizer_config.json","special_tokens_map.json")
    hashes={n:file_hash(path/n) for n in names if (path/n).is_file()}
    require('tokenizer.json' in hashes,'Missing tokenizer.json')
    return {'files':hashes,'sha256':json_hash(hashes)}

class PackedDataset(Dataset):
    def __init__(self,directory,split,block_length=2048,verify_hashes=False):
        self.root=Path(directory).resolve(); self.manifest=read_json(self.root/'token_manifest.json')
        check=dict(self.manifest); checksum=check.pop('manifest_hash')
        require(json_hash(check)==checksum,'Token manifest checksum mismatch')
        require(split in ('train','validation'),'Invalid split')
        require(block_length>=2,'Invalid block length')
        self.block_length=int(block_length); self.dtype=np.dtype(self.manifest['dtype'])
        self.streams=[]; self.ends=[]; self.maps={}; total=0
        for stream in self.manifest['streams']:
            if stream['split']!=split: continue
            path=self.root/stream['file']
            require(path.is_file(),f'Missing stream {path}')
            require(path.stat().st_size==stream['bytes'],f'Size mismatch {path.name}')
            if verify_hashes: require(file_hash(path)==stream['sha256'],f'Hash mismatch {path.name}')
            blocks=int(stream['tokens'])//self.block_length
            if blocks:
                self.streams.append(stream); total+=blocks; self.ends.append(total)
        require(total>0,f'No complete {split} blocks')
    def __len__(self): return self.ends[-1]
    def __getitem__(self,index):
        if index<0: index+=len(self)
        require(0<=index<len(self),'Block index out of range')
        fi=bisect.bisect_right(self.ends,index); prior=self.ends[fi-1] if fi else 0
        if fi not in self.maps:
            self.maps[fi]=np.memmap(self.root/self.streams[fi]['file'],dtype=self.dtype,mode='r')
        start=(index-prior)*self.block_length
        view=self.maps[fi][start:start+self.block_length]
        return {'input_ids':torch.from_numpy(np.asarray(view,dtype=np.int64).copy())}
    def __getstate__(self):
        d=self.__dict__.copy(); d['maps']={}; return d
    def close(self):
        for m in self.maps.values():
            try:m._mmap.close()
            except Exception:pass
        self.maps.clear()
    def __del__(self):
        try:self.close()
        except Exception:pass

class EncoderDecoderView(Dataset):
    def __init__(self,packed,source_length,target_length,bos_token_id):
        self.packed=packed; self.source_length=int(source_length); self.target_length=int(target_length); self.bos_token_id=int(bos_token_id)
        require(self.source_length+self.target_length==packed.block_length,'Split must equal packed block')
    def __len__(self): return len(self.packed)
    def __getitem__(self,index):
        block=self.packed[index]['input_ids']; source=block[:self.source_length]; labels=block[self.source_length:]
        dec=torch.empty_like(labels); dec[0]=self.bos_token_id; dec[1:]=labels[:-1]
        return {'source_ids':source,'decoder_input_ids':dec,'labels':labels}
    def close(self): self.packed.close()

class ResumableBatches(Sampler):
    def __init__(self,size,batch_size,start_microbatch,microbatches,seed):
        self.size=int(size); self.batch_size=int(batch_size); self.start=int(start_microbatch); self.count=int(microbatches); self.seed=int(seed)
        require(self.size>0 and self.batch_size>0 and self.count>=0,'Invalid sampler dimensions')
    def __len__(self): return self.count
    def __iter__(self):
        epoch_cache=None; a,b=1,0
        for micro in range(self.start,self.start+self.count):
            indices=[]
            for absolute in range(micro*self.batch_size,(micro+1)*self.batch_size):
                epoch,offset=divmod(absolute,self.size)
                if epoch!=epoch_cache:
                    rng=random.Random(f'{self.seed}:block-order:{epoch}')
                    a=rng.randrange(1,self.size+1)
                    while math.gcd(a,self.size)!=1: a=a%self.size+1
                    b=rng.randrange(self.size); epoch_cache=epoch
                indices.append((a*offset+b)%self.size)
            yield indices

class EvenlySpacedSubset(Dataset):
    def __init__(self,dataset,count): self.dataset=dataset; self.count=int(count)
    def __len__(self): return self.count
    def __getitem__(self,index): return self.dataset[index*len(self.dataset)//self.count]

def seed_worker(worker_id):
    seed=torch.initial_seed()%2**32; np.random.seed(seed); random.seed(seed)

def loader_options(cfg):
    workers=int(cfg['num_workers']); require(workers>=0,'num_workers must be nonnegative')
    out={'num_workers':workers,'pin_memory':bool(cfg['pin_memory']),'worker_init_fn':seed_worker,'generator':torch.Generator().manual_seed(int(cfg['seed']))}
    if workers:
        out.update(persistent_workers=bool(cfg['persistent_workers']),prefetch_factor=int(cfg['prefetch_factor']))
    return out

Writing legal_encdec_data.py


In [9]:
import importlib
import legal_encdec_data as encdata
importlib.reload(encdata)

train_packed = encdata.PackedDataset(
    TOKEN_STORE, "train", block_length=RAW_BLOCK_LENGTH, verify_hashes=VERIFY_TOKEN_HASHES
)
val_packed = encdata.PackedDataset(
    TOKEN_STORE, "validation", block_length=RAW_BLOCK_LENGTH, verify_hashes=VERIFY_TOKEN_HASHES
)
train_data = encdata.EncoderDecoderView(train_packed, SOURCE_LENGTH, TARGET_LENGTH, tokenizer.bos_token_id)
val_data = encdata.EncoderDecoderView(val_packed, SOURCE_LENGTH, TARGET_LENGTH, tokenizer.bos_token_id)

sample = train_data[0]
print("Train blocks:", len(train_data))
print("Validation blocks:", len(val_data))
print({k: tuple(v.shape) for k, v in sample.items()})
assert sample["source_ids"].shape == (1536,)
assert sample["decoder_input_ids"].shape == (512,)
assert sample["labels"].shape == (512,)
assert sample["decoder_input_ids"][0].item() == tokenizer.bos_token_id
assert torch.equal(sample["decoder_input_ids"][1:], sample["labels"][:-1])
print("Dataset view passed.")

Train blocks: 97701
Validation blocks: 2457
{'source_ids': (1536,), 'decoder_input_ids': (512,), 'labels': (512,)}
Dataset view passed.


## Final Model B configuration


In [10]:
MODEL_CONFIG = dict(
    vocab_size=VOCAB_SIZE,
    source_length=SOURCE_LENGTH,
    target_length=TARGET_LENGTH,
    d_model=640,
    n_heads=10,
    n_encoder_layers=5,
    n_decoder_layers=5,
    d_ff=2560,
    dropout=0.1,
    norm_epsilon=1e-5,
    tie_embeddings=True,
    activation_checkpointing=False,  # recorded for the experiment; model intentionally does not use checkpointing
)

assert MODEL_CONFIG["d_model"] % MODEL_CONFIG["n_heads"] == 0
print(MODEL_CONFIG)
print("Head dim:", MODEL_CONFIG["d_model"] // MODEL_CONFIG["n_heads"])

{'vocab_size': 32000, 'source_length': 1536, 'target_length': 512, 'd_model': 640, 'n_heads': 10, 'n_encoder_layers': 5, 'n_decoder_layers': 5, 'd_ff': 2560, 'dropout': 0.1, 'norm_epsilon': 1e-05, 'tie_embeddings': True, 'activation_checkpointing': False}
Head dim: 64


## Model B architecture


In [11]:
%%writefile legal_encdec_model.py
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


def parameter_counts(model):
    seen = set(); total = 0; trainable = 0
    for p in model.parameters():
        if id(p) in seen:
            continue
        seen.add(id(p)); total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    return {"total": total, "trainable": trainable}


def seq2seq_loss(logits, labels):
    if logits.ndim != 3 or labels.ndim != 2 or logits.shape[:2] != labels.shape:
        raise ValueError("Expected logits [B,T,V] and labels [B,T]")
    return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1))


def initialize_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
    elif isinstance(module, nn.LayerNorm):
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["d_model"], cfg["d_ff"])
        self.fc2 = nn.Linear(cfg["d_ff"], cfg["d_model"])
        self.dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return self.dropout(x)


class EncoderSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.d_model = cfg["d_model"]
        self.n_heads = cfg["n_heads"]
        if self.d_model % self.n_heads:
            raise ValueError("d_model must be divisible by n_heads")
        self.head_dim = self.d_model // self.n_heads
        self.dropout_p = float(cfg["dropout"])
        self.qkv = nn.Linear(self.d_model, 3 * self.d_model)
        self.out = nn.Linear(self.d_model, self.d_model)
        self.out_dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        B, S, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1, 2).contiguous().view(B, S, D)
        return self.out_dropout(self.out(y))


class DecoderSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.d_model = cfg["d_model"]
        self.n_heads = cfg["n_heads"]
        if self.d_model % self.n_heads:
            raise ValueError("d_model must be divisible by n_heads")
        self.head_dim = self.d_model // self.n_heads
        self.dropout_p = float(cfg["dropout"])
        self.qkv = nn.Linear(self.d_model, 3 * self.d_model)
        self.out = nn.Linear(self.d_model, self.d_model)
        self.out_dropout = nn.Dropout(cfg["dropout"])

    def _shape(self, x, B, T):
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = self._shape(q, B, T); k = self._shape(k, B, T); v = self._shape(v, B, T)
        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=True,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_dropout(self.out(y))

    def step(self, x, past_kv=None):
        # x: [B,1,D]. There are no future keys in the cache, so no causal mask is needed.
        B, T, D = x.shape
        if T != 1:
            raise ValueError("DecoderSelfAttention.step expects one token")
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = self._shape(q, B, 1); k = self._shape(k, B, 1); v = self._shape(v, B, 1)
        if past_kv is not None:
            pk, pv = past_kv
            k = torch.cat((pk, k), dim=-2)
            v = torch.cat((pv, v), dim=-2)
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=False)
        y = y.transpose(1, 2).contiguous().view(B, 1, D)
        return self.out(y), (k, v)


class CrossAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.d_model = cfg["d_model"]
        self.n_heads = cfg["n_heads"]
        if self.d_model % self.n_heads:
            raise ValueError("d_model must be divisible by n_heads")
        self.head_dim = self.d_model // self.n_heads
        self.dropout_p = float(cfg["dropout"])
        self.q_proj = nn.Linear(self.d_model, self.d_model)
        self.kv_proj = nn.Linear(self.d_model, 2 * self.d_model)
        self.out = nn.Linear(self.d_model, self.d_model)
        self.out_dropout = nn.Dropout(cfg["dropout"])

    def _shape(self, x, B, L):
        return x.view(B, L, self.n_heads, self.head_dim).transpose(1, 2)

    def project_kv(self, memory):
        B, S, _ = memory.shape
        k, v = self.kv_proj(memory).chunk(2, dim=-1)
        return self._shape(k, B, S), self._shape(v, B, S)

    def forward_projected(self, x, projected_kv):
        B, T, D = x.shape
        k, v = projected_kv
        q = self._shape(self.q_proj(x), B, T)
        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_dropout(self.out(y))

    def forward(self, x, memory):
        return self.forward_projected(x, self.project_kv(memory))


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg["d_model"]; eps = cfg["norm_epsilon"]
        self.norm1 = nn.LayerNorm(d, eps=eps)
        self.self_attention = EncoderSelfAttention(cfg)
        self.norm2 = nn.LayerNorm(d, eps=eps)
        self.ffn = FeedForward(cfg)

    def forward(self, x):
        x = x + self.self_attention(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg["d_model"]; eps = cfg["norm_epsilon"]
        self.norm1 = nn.LayerNorm(d, eps=eps)
        self.self_attention = DecoderSelfAttention(cfg)
        self.norm2 = nn.LayerNorm(d, eps=eps)
        self.cross_attention = CrossAttention(cfg)
        self.norm3 = nn.LayerNorm(d, eps=eps)
        self.ffn = FeedForward(cfg)

    def forward(self, x, memory):
        x = x + self.self_attention(self.norm1(x))
        x = x + self.cross_attention(self.norm2(x), memory)
        x = x + self.ffn(self.norm3(x))
        return x

    def prepare_cross(self, memory):
        return self.cross_attention.project_kv(memory)

    def step(self, x, past_kv, cross_kv):
        sa, new_kv = self.self_attention.step(self.norm1(x), past_kv)
        x = x + sa
        x = x + self.cross_attention.forward_projected(self.norm2(x), cross_kv)
        x = x + self.ffn(self.norm3(x))
        return x, new_kv


class EncoderDecoderLegalLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = dict(cfg)
        required = (
            "vocab_size", "source_length", "target_length", "d_model", "n_heads",
            "n_encoder_layers", "n_decoder_layers", "d_ff", "dropout",
            "norm_epsilon", "tie_embeddings",
        )
        for key in required:
            if key not in cfg:
                raise ValueError(f"Missing config field: {key}")
        if cfg["d_model"] % cfg["n_heads"]:
            raise ValueError("d_model must be divisible by n_heads")
        if min(cfg["source_length"], cfg["target_length"], cfg["n_encoder_layers"], cfg["n_decoder_layers"]) <= 0:
            raise ValueError("Lengths/layer counts must be positive")

        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.encoder_positions = nn.Embedding(cfg["source_length"], cfg["d_model"])
        self.decoder_positions = nn.Embedding(cfg["target_length"], cfg["d_model"])
        self.embedding_dropout = nn.Dropout(cfg["dropout"])
        self.encoder_blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg["n_encoder_layers"])])
        self.decoder_blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg["n_decoder_layers"])])
        self.encoder_norm = nn.LayerNorm(cfg["d_model"], eps=cfg["norm_epsilon"])
        self.decoder_norm = nn.LayerNorm(cfg["d_model"], eps=cfg["norm_epsilon"])
        self.lm_head = nn.Linear(cfg["d_model"], cfg["vocab_size"], bias=False)
        self.apply(initialize_weights)
        if cfg["tie_embeddings"]:
            self.lm_head.weight = self.token_embedding.weight

    def encode(self, source_ids):
        if source_ids.ndim != 2:
            raise ValueError("source_ids must be [B,S]")
        B, S = source_ids.shape
        if not 0 < S <= self.cfg["source_length"]:
            raise ValueError("Invalid source length")
        pos = torch.arange(S, device=source_ids.device)
        x = self.token_embedding(source_ids) + self.encoder_positions(pos)[None, :, :]
        x = self.embedding_dropout(x)
        for block in self.encoder_blocks:
            x = block(x)
        return self.encoder_norm(x)

    def decode(self, decoder_input_ids, memory):
        if decoder_input_ids.ndim != 2:
            raise ValueError("decoder_input_ids must be [B,T]")
        B, T = decoder_input_ids.shape
        if not 0 < T <= self.cfg["target_length"]:
            raise ValueError("Invalid decoder length")
        if memory.ndim != 3 or memory.shape[0] != B:
            raise ValueError("Invalid encoder memory")
        pos = torch.arange(T, device=decoder_input_ids.device)
        x = self.token_embedding(decoder_input_ids) + self.decoder_positions(pos)[None, :, :]
        x = self.embedding_dropout(x)
        for block in self.decoder_blocks:
            x = block(x, memory)
        return self.decoder_norm(x)

    def forward(self, source_ids, decoder_input_ids):
        if source_ids.shape[0] != decoder_input_ids.shape[0]:
            raise ValueError("Batch mismatch")
        memory = self.encode(source_ids)
        return self.lm_head(self.decode(decoder_input_ids, memory))

    @torch.inference_mode()
    def prepare_generation(self, source_ids):
        was_training = self.training
        self.eval()
        memory = self.encode(source_ids)
        cross_cache = [block.prepare_cross(memory) for block in self.decoder_blocks]
        if was_training:
            self.train()
        return cross_cache

    @torch.inference_mode()
    def generation_step(self, token_ids, position, cross_cache, self_cache=None):
        # token_ids: [B,1], position is the absolute decoder position of token_ids.
        if token_ids.ndim != 2 or token_ids.shape[1] != 1:
            raise ValueError("generation_step expects [B,1]")
        if not 0 <= int(position) < self.cfg["target_length"]:
            raise ValueError("Decoder position outside trained range")
        was_training = self.training
        self.eval()
        B = token_ids.shape[0]
        pos = torch.tensor([int(position)], device=token_ids.device)
        x = self.token_embedding(token_ids) + self.decoder_positions(pos)[None, :, :]
        # dropout is disabled in eval
        if self_cache is None:
            self_cache = [None] * len(self.decoder_blocks)
        if len(self_cache) != len(self.decoder_blocks) or len(cross_cache) != len(self.decoder_blocks):
            raise ValueError("Cache length mismatch")
        new_cache = []
        for i, block in enumerate(self.decoder_blocks):
            x, kv = block.step(x, self_cache[i], cross_cache[i])
            new_cache.append(kv)
        logits = self.lm_head(self.decoder_norm(x))
        if was_training:
            self.train()
        return logits, new_cache

Writing legal_encdec_model.py


## Architecture + backward + cache contract tests


In [12]:
import importlib
import legal_encdec_model as encmodel
importlib.reload(encmodel)

EncoderDecoderLegalLM = encmodel.EncoderDecoderLegalLM
seq2seq_loss = encmodel.seq2seq_loss
parameter_counts = encmodel.parameter_counts

# CPU parameter summary avoids Kaggle's current meta-device initialization bug.
summary_model = EncoderDecoderLegalLM(MODEL_CONFIG)
counts = parameter_counts(summary_model)
print("Parameter counts:", counts)
print("Parameters (M):", round(counts["total"] / 1e6, 2))
assert 50_000_000 <= counts["total"] <= 300_000_000
assert summary_model.lm_head.weight is summary_model.token_embedding.weight
del summary_model
gc.collect()

# Real forward/backward smoke test.
contract_model = EncoderDecoderLegalLM(MODEL_CONFIG).to(DEVICE)
contract_model.eval()
B, S, T = 1, 64, 16
src = torch.randint(4, VOCAB_SIZE, (B, S), device=DEVICE)
dec = torch.randint(4, VOCAB_SIZE, (B, T), device=DEVICE)

with torch.no_grad():
    logits = contract_model(src, dec)
assert logits.shape == (B, T, VOCAB_SIZE)
assert torch.isfinite(logits).all()

# Causality.
cut = 7
dec_changed = dec.clone()
dec_changed[:, cut + 1:] = torch.randint(4, VOCAB_SIZE, dec_changed[:, cut + 1:].shape, device=DEVICE)
with torch.no_grad():
    a = contract_model(src, dec)
    b = contract_model(src, dec_changed)
torch.testing.assert_close(a[:, :cut + 1], b[:, :cut + 1], rtol=1e-4, atol=1e-5)

# Source conditioning.
src2 = torch.randint(4, VOCAB_SIZE, (B, S), device=DEVICE)
with torch.no_grad():
    c = contract_model(src2, dec)
assert not torch.allclose(a, c), "Cross-attention appears disconnected from encoder source"

# Backward.
contract_model.train()
labels = torch.randint(4, VOCAB_SIZE, (B, T), device=DEVICE)
loss = seq2seq_loss(contract_model(src, dec), labels)
assert torch.isfinite(loss)
loss.backward()
assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in contract_model.parameters() if p.requires_grad)
print("Backward passed. Loss:", float(loss.detach()))
contract_model.zero_grad(set_to_none=True)

# Cached generation must match full decoder evaluation in eval mode.
contract_model.eval()
with torch.no_grad():
    mem = contract_model.encode(src)
    cross_cache = [block.prepare_cross(mem) for block in contract_model.decoder_blocks]
    self_cache = None
    prefix = dec[:, :5]
    for pos in range(prefix.shape[1]):
        step_logits, self_cache = contract_model.generation_step(
            prefix[:, pos:pos+1], pos, cross_cache, self_cache
        )
        full_logits = contract_model.lm_head(contract_model.decode(prefix[:, :pos+1], mem))[:, -1:]
        torch.testing.assert_close(step_logits, full_logits, rtol=1e-4, atol=1e-5)

print("Architecture contract tests passed.")
del contract_model, src, src2, dec, dec_changed, labels, logits, a, b, c, loss
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Parameter counts: {'total': 79239680, 'trainable': 79239680}
Parameters (M): 79.24
Backward passed. Loss: 10.535493850708008
Architecture contract tests passed.


## Two-GPU training module


In [13]:
%%writefile legal_encdec_training.py
from __future__ import annotations
import contextlib, hashlib, json, math, os, random, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import legal_encdec_data as data
from legal_encdec_model import seq2seq_loss, parameter_counts

class ReplicaAutocast(nn.Module):
    def __init__(self, model, dtype):
        super().__init__(); self.model=model; self.dtype=dtype
    @property
    def cfg(self): return self.model.cfg
    def forward(self,*args,**kwargs):
        # This forward runs inside each DataParallel replica thread, where autocast state
        # from the main thread is not guaranteed to propagate.
        with torch.autocast(device_type='cuda', dtype=self.dtype, enabled=True):
            return self.model(*args,**kwargs)

class ConfigDataParallel(nn.DataParallel):
    @property
    def cfg(self): return self.module.cfg

def unwrap_model(model):
    m=model
    while True:
        if isinstance(m, nn.DataParallel): m=m.module; continue
        if isinstance(m, ReplicaAutocast): m=m.model; continue
        return m

def precision_for(device,requested='auto'):
    if requested=='auto': return 'bf16' if device.type=='cuda' and torch.cuda.is_bf16_supported() else ('fp16' if device.type=='cuda' else 'fp32')
    if requested not in ('fp32','fp16','bf16'): raise ValueError('Invalid precision')
    return requested

def amp(device,precision):
    if precision=='fp32': return contextlib.nullcontext()
    dtype=torch.bfloat16 if precision=='bf16' else torch.float16
    return torch.autocast(device_type=device.type,dtype=dtype)

def prepare_training_model(core_model, precision, device_ids=(0,1)):
    if not torch.cuda.is_available():
        return core_model
    ids=[int(i) for i in device_ids if int(i)<torch.cuda.device_count()]
    if len(ids)<2:
        return core_model
    if precision=='fp32':
        wrapped=core_model
    else:
        dtype=torch.bfloat16 if precision=='bf16' else torch.float16
        wrapped=ReplicaAutocast(core_model,dtype)
    return ConfigDataParallel(wrapped,device_ids=ids,output_device=ids[0])

def seed_all(seed,strict=False):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(bool(strict))

def plan(mc,tc):
    raw_block=int(mc['source_length']+mc['target_length']); target_per=int(mc['target_length']); raw_per=int(tc['raw_tokens_per_optimizer_step'])
    if raw_per%raw_block: raise ValueError('raw_tokens_per_optimizer_step must be divisible by block length')
    blocks=raw_per//raw_block; bs=int(tc['physical_batch_size'])
    if blocks%bs: raise ValueError('physical_batch_size must divide blocks_per_optimizer_step')
    steps=int(tc['train_raw_token_budget'])//raw_per
    if steps<=0: raise ValueError('Training budget too small')
    return {'raw_block_tokens':raw_block,'target_tokens_per_block':target_per,'blocks_per_optimizer_step':blocks,'physical_batch_size':bs,'gradient_accumulation_steps':blocks//bs,'raw_tokens_per_step':raw_per,'target_tokens_per_step':blocks*target_per,'steps':steps,'warmup':math.ceil(steps*float(tc['warmup_ratio'])),'planned_raw_tokens':steps*raw_per,'planned_target_tokens':steps*blocks*target_per}

def autotune_microbatch(model,tc,device):
    cfg=dict(tc); precision=precision_for(device,cfg['precision']); mc=model.cfg
    blocks=cfg['raw_tokens_per_optimizer_step']//(mc['source_length']+mc['target_length'])
    candidates=[int(x) for x in cfg['microbatch_candidates'] if int(x)>=2 and blocks%int(x)==0]
    if not candidates: raise ValueError('No >=2 candidate divides optimizer block count')
    was=model.training; model.train()
    try:
        for bs in candidates:
            try:
                model.zero_grad(set_to_none=True)
                if device.type=='cuda':
                    for i in range(torch.cuda.device_count()): torch.cuda.reset_peak_memory_stats(i)
                    torch.cuda.empty_cache()
                src=torch.randint(4,mc['vocab_size'],(bs,mc['source_length']),device=device)
                dec=torch.randint(4,mc['vocab_size'],(bs,mc['target_length']),device=device)
                labels=torch.randint(4,mc['vocab_size'],(bs,mc['target_length']),device=device)
                with amp(device,precision):
                    loss=seq2seq_loss(model(src,dec),labels)
                loss.backward()
                peaks=[]
                if device.type=='cuda':
                    peaks=[round(torch.cuda.max_memory_allocated(i)/2**30,2) for i in range(torch.cuda.device_count())]
                cfg['physical_batch_size']=bs; cfg['gradient_accumulation_steps']=blocks//bs; cfg['eval_batch_size']=bs
                print({'fit_probe_global_batch':bs,'grad_accum':cfg['gradient_accumulation_steps'],'peak_allocated_gib':peaks})
                del src,dec,labels,loss; model.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                return cfg
            except torch.cuda.OutOfMemoryError:
                print('OOM in fit probe at global batch',bs)
                model.zero_grad(set_to_none=True); torch.cuda.empty_cache()
        raise RuntimeError('Global batch 2 did not fit across the two GPUs')
    finally:
        model.train(was)

def optimizer_scheduler(model,tc,p):
    decay=[]; no_decay=[]
    for param in model.parameters(): (decay if param.ndim>=2 else no_decay).append(param)
    kwargs=dict(lr=tc['learning_rate'],betas=tuple(tc['betas']),eps=tc['epsilon'])
    if tc.get('fused_optimizer',False) and next(model.parameters()).is_cuda: kwargs['fused']=True
    groups=[{'params':decay,'weight_decay':tc['weight_decay']},{'params':no_decay,'weight_decay':0.0}]
    try: opt=torch.optim.AdamW(groups,**kwargs)
    except (TypeError,RuntimeError): kwargs.pop('fused',None); opt=torch.optim.AdamW(groups,**kwargs)
    def fn(step):
        if p['warmup'] and step<p['warmup']: return (step+1)/p['warmup']
        progress=min(1.0,(step-p['warmup'])/max(1,p['steps']-p['warmup']))
        return 0.5*(1+math.cos(math.pi*progress)) if tc['scheduler']=='cosine' else (1-progress if tc['scheduler']=='linear' else 1.0)
    return opt,torch.optim.lr_scheduler.LambdaLR(opt,fn)

@torch.no_grad()
def validate(model,dataset,raw_budget,batch_size,tc,device,precision):
    mc=model.cfg; raw_block=mc['source_length']+mc['target_length']; count=min(len(dataset),int(raw_budget)//raw_block)
    if count<=0: raise ValueError('Validation budget too small')
    subset=data.EvenlySpacedSubset(dataset,count)
    loader=DataLoader(subset,batch_size=int(batch_size),**data.loader_options(tc))
    prior=model.training; model.eval(); total=0.0; targets=0
    try:
        for batch in loader:
            src=batch['source_ids'].to(device,non_blocking=True); dec=batch['decoder_input_ids'].to(device,non_blocking=True); labels=batch['labels'].to(device,non_blocking=True)
            with amp(device,precision): loss=seq2seq_loss(model(src,dec),labels)
            if not torch.isfinite(loss): raise RuntimeError('Nonfinite validation loss')
            n=labels.numel(); total+=float(loss)*n; targets+=n
    finally: model.train(prior)
    mean=total/targets
    return {'validation_loss':mean,'validation_perplexity':math.exp(mean) if mean<700 else None,'validation_targets':targets,'validation_raw_tokens':count*raw_block}

def rng_state(): return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}
def restore_rng(r):
    random.setstate(r['python']); np.random.set_state(r['numpy']); torch.set_rng_state(r['torch'])
    if r['cuda'] is not None: torch.cuda.set_rng_state_all(r['cuda'])
def json_hash(obj): return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
def file_hash(path,block_size=8*1024*1024):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(block_size),b''): h.update(b)
    return h.hexdigest()
def write_json(path,value):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); tmp=path.with_name(path.name+'.tmp')
    with tmp.open('w',encoding='utf-8') as f: json.dump(value,f,indent=2,ensure_ascii=False,allow_nan=False); f.flush(); os.fsync(f.fileno())
    os.replace(tmp,path)

def identity(model,tc,dataset,tokenizer_path):
    core=unwrap_model(model); here=Path(__file__).resolve().parent
    implementation={name:file_hash(here/name) for name in ('legal_encdec_model.py','legal_encdec_data.py','legal_encdec_training.py')}
    return {'model':core.cfg,'training':tc,'dataset_manifest':dataset.packed.manifest['manifest_hash'],'tokenizer':data.tokenizer_identity(tokenizer_path)['sha256'],'implementation':implementation}

def save_checkpoint(root,model,opt,sched,scaler,state,ident,tc):
    core=unwrap_model(model); target=root/f"checkpoint-{state['step']:08d}.pt"; tmp=root/(target.name+'.partial')
    if target.exists(): raise RuntimeError(f'Refusing to overwrite {target}')
    with tmp.open('wb') as f:
        torch.save({'model':core.state_dict(),'optimizer':opt.state_dict(),'scheduler':sched.state_dict(),'scaler':scaler.state_dict(),'state':state,'identity':ident,'rng':rng_state()},f); f.flush(); os.fsync(f.fileno())
    os.replace(tmp,target)
    owned=sorted(root.glob('checkpoint-????????.pt'))
    for old in owned[:-int(tc['keep_checkpoints'])]: old.unlink()
    return target

def train(model,train_data,val_data,tc,output_dir,tokenizer_path,run_name,resume_from=None,stop_after_step=None):
    device=next(model.parameters()).device; precision=precision_for(device,tc['precision']); p=plan(model.cfg,tc); root=Path(output_dir)/run_name; root.mkdir(parents=True,exist_ok=True)
    opt,sched=optimizer_scheduler(model,tc,p); scaler=torch.amp.GradScaler('cuda',enabled=(device.type=='cuda' and precision=='fp16')); ident=identity(model,tc,train_data,tokenizer_path)
    state={'step':0,'raw_tokens':0,'target_tokens':0,'cursor':0,'elapsed':0.0,'validation':{}}
    if resume_from is not None:
        saved=torch.load(resume_from,map_location='cpu',weights_only=False)
        if saved['identity']!=ident: raise RuntimeError('Config/data/code changed; exact resume refused')
        unwrap_model(model).load_state_dict(saved['model']); opt.load_state_dict(saved['optimizer']); sched.load_state_dict(saved['scheduler']); scaler.load_state_dict(saved['scaler']); state=saved['state']; restore_rng(saved['rng'])
    else:
        if list(root.glob('checkpoint-*')) or (root/'training.jsonl').exists(): raise RuntimeError('Existing run found; resume it or use a fresh RUN_NAME')
        seed_all(tc['seed'],tc.get('strict_determinism',False))
    if state['raw_tokens']!=state['step']*p['raw_tokens_per_step'] or state['target_tokens']!=state['step']*p['target_tokens_per_step']: raise RuntimeError('Checkpoint token accounting mismatch')
    target=p['steps'] if stop_after_step is None else min(int(stop_after_step),p['steps'])
    if target<=state['step']: raise ValueError('Target step must exceed current step')
    write_json(root/'model_config.json',unwrap_model(model).cfg); write_json(root/'training_config.json',tc); write_json(root/'training_manifest.json',{'run_name':run_name,'parameters':parameter_counts(unwrap_model(model)),'plan':p,'identity':ident,'gpu_count':torch.cuda.device_count()})
    microbatches=(target-state['step'])*p['gradient_accumulation_steps']; sampler=data.ResumableBatches(len(train_data),tc['physical_batch_size'],state['cursor'],microbatches,tc['seed'])
    batches=iter(DataLoader(train_data,batch_sampler=sampler,**data.loader_options(tc)))
    start=time.perf_counter(); old_elapsed=state['elapsed']; start_raw=state['raw_tokens']; latest=None; model.train()
    with (root/'training.jsonl').open('a',encoding='utf-8',buffering=1) as log:
        while state['step']<target:
            opt.zero_grad(set_to_none=True); loss_sum=torch.zeros((),device=device); lr=opt.param_groups[0]['lr']
            for _ in range(p['gradient_accumulation_steps']):
                batch=next(batches); src=batch['source_ids'].to(device,non_blocking=True); dec=batch['decoder_input_ids'].to(device,non_blocking=True); labels=batch['labels'].to(device,non_blocking=True)
                with amp(device,precision): loss=seq2seq_loss(model(src,dec),labels)
                if not torch.isfinite(loss): raise RuntimeError('Nonfinite training loss')
                loss_sum+=loss.detach(); scaler.scale(loss/p['gradient_accumulation_steps']).backward()
            scaler.unscale_(opt); norm=torch.nn.utils.clip_grad_norm_(model.parameters(),tc['max_grad_norm'],error_if_nonfinite=True); scaler.step(opt); scaler.update(); sched.step()
            state['step']+=1; state['cursor']+=p['gradient_accumulation_steps']; state['raw_tokens']+=p['raw_tokens_per_step']; state['target_tokens']+=p['target_tokens_per_step']
            is_final=state['step']==target; should_eval=(state['step']%tc['eval_steps']==0 or is_final)
            if should_eval:
                budget=tc['validation_raw_token_budget'] if is_final else tc['eval_raw_token_budget']; state['validation']=validate(model,val_data,budget,tc['eval_batch_size'],tc,device,precision)
            elapsed_session=time.perf_counter()-start; state['elapsed']=old_elapsed+elapsed_session
            should_log=(state['step']%tc['logging_steps']==0 or should_eval or state['step']%tc['checkpoint_steps']==0)
            if should_log:
                metrics={'step':state['step'],'raw_tokens':state['raw_tokens'],'target_tokens':state['target_tokens'],'loss':float((loss_sum/p['gradient_accumulation_steps']).item()),'learning_rate':lr,'gradient_norm':float(norm.item() if torch.is_tensor(norm) else norm),'elapsed':state['elapsed'],'raw_tokens_per_second':(state['raw_tokens']-start_raw)/max(elapsed_session,1e-9)}
                if should_eval: metrics.update(state['validation'])
                log.write(json.dumps(metrics,allow_nan=False)+'\n'); print(metrics,flush=True)
            if state['step']%tc['checkpoint_steps']==0 or is_final:
                latest=save_checkpoint(root,model,opt,sched,scaler,state,ident,tc); log.flush(); os.fsync(log.fileno()); write_json(root/'progress.json',state); write_json(root/'ablation_result.json',{'run_name':run_name,'parameter_count':parameter_counts(unwrap_model(model))['total'],'raw_tokens_processed':state['raw_tokens'],'target_tokens_processed':state['target_tokens'],'checkpoint':str(latest),**state['validation']})
    return {'model':model,'state':state,'checkpoint':str(latest)}

Writing legal_encdec_training.py


## Training configuration — **50M raw tokens**


In [14]:
TRAIN_CONFIG = dict(
    raw_tokens_per_optimizer_step=65_536,
    auto_microbatch=True,
    microbatch_candidates=[4, 2],  # global batches; with 2 GPUs => 2 or 1 example/GPU
    physical_batch_size=2,
    gradient_accumulation_steps=16,
    eval_batch_size=2,

    learning_rate=2e-4,
    weight_decay=0.1,
    warmup_ratio=0.02,
    max_grad_norm=1.0,

    train_raw_token_budget=50_000_000,
    validation_raw_token_budget=1_000_000,
    eval_raw_token_budget=500_000,

    logging_steps=25,
    eval_steps=250,
    checkpoint_steps=125,
    keep_checkpoints=2,

    fused_optimizer=True,
    scheduler="cosine",
    betas=[0.9, 0.95],
    epsilon=1e-8,
    precision="auto",
    strict_determinism=False,
    seed=SEED,

    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

print(json.dumps(TRAIN_CONFIG, indent=2))

{
  "raw_tokens_per_optimizer_step": 65536,
  "auto_microbatch": true,
  "microbatch_candidates": [
    4,
    2
  ],
  "physical_batch_size": 2,
  "gradient_accumulation_steps": 16,
  "eval_batch_size": 2,
  "learning_rate": 0.0002,
  "weight_decay": 0.1,
  "warmup_ratio": 0.02,
  "max_grad_norm": 1.0,
  "train_raw_token_budget": 50000000,
  "validation_raw_token_budget": 1000000,
  "eval_raw_token_budget": 500000,
  "logging_steps": 25,
  "eval_steps": 250,
  "checkpoint_steps": 125,
  "keep_checkpoints": 2,
  "fused_optimizer": true,
  "scheduler": "cosine",
  "betas": [
    0.9,
    0.95
  ],
  "epsilon": 1e-08,
  "precision": "auto",
  "strict_determinism": false,
  "seed": 42,
  "num_workers": 2,
  "pin_memory": true,
  "persistent_workers": true,
  "prefetch_factor": 4
}


## 2-GPU preflight and training plan


In [15]:
import importlib
import legal_encdec_training as training
importlib.reload(training)

assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."
assert torch.cuda.device_count() >= 2, "Select Kaggle T4 x2. This pipeline is configured to use both GPUs."

training.seed_all(SEED, False)
core_probe = EncoderDecoderLegalLM(MODEL_CONFIG).to("cuda:0")
precision = training.precision_for(torch.device("cuda:0"), TRAIN_CONFIG["precision"])
probe_model = training.prepare_training_model(core_probe, precision, device_ids=(0, 1))

if AUTO_MICROBATCH:
    TRAIN_CONFIG = training.autotune_microbatch(probe_model, TRAIN_CONFIG, torch.device("cuda:0"))

plan = training.plan(MODEL_CONFIG, TRAIN_CONFIG)
print()
print("FINAL TRAINING PLAN")
print(json.dumps(plan, indent=2))
print("Global physical batch:", TRAIN_CONFIG["physical_batch_size"])
print("Approx examples/GPU:", TRAIN_CONFIG["physical_batch_size"] // 2)
print("Gradient accumulation:", plan["gradient_accumulation_steps"])
print("Planned raw tokens:", plan["planned_raw_tokens"])
print("Planned target tokens:", plan["planned_target_tokens"])
assert plan["steps"] == 762
assert plan["planned_raw_tokens"] <= 50_000_000
assert TRAIN_CONFIG["physical_batch_size"] >= 2

required_blocks = plan["steps"] * plan["blocks_per_optimizer_step"]
assert len(train_data) >= required_blocks

del probe_model, core_probe
gc.collect()
torch.cuda.empty_cache()

{'fit_probe_global_batch': 4, 'grad_accum': 8, 'peak_allocated_gib': [4.69, 4.13]}

FINAL TRAINING PLAN
{
  "raw_block_tokens": 2048,
  "target_tokens_per_block": 512,
  "blocks_per_optimizer_step": 32,
  "physical_batch_size": 4,
  "gradient_accumulation_steps": 8,
  "raw_tokens_per_step": 65536,
  "target_tokens_per_step": 16384,
  "steps": 762,
  "warmup": 16,
  "planned_raw_tokens": 49938432,
  "planned_target_tokens": 12484608
}
Global physical batch: 4
Approx examples/GPU: 2
Gradient accumulation: 8
Planned raw tokens: 49938432
Planned target tokens: 12484608


## Training and resume


In [16]:
result = None
FINAL_CHECKPOINT = None

if RUN_FULL_TRAINING:
    assert torch.cuda.device_count() >= 2
    run_dir = OUTPUT_DIR / RUN_NAME
    checkpoints = sorted(run_dir.glob("checkpoint-????????.pt")) if run_dir.exists() else []

    # Archive an incomplete run that has no resumable checkpoint.
    if not checkpoints and run_dir.exists() and any(run_dir.iterdir()):
        archive = OUTPUT_DIR / f"{RUN_NAME}.incomplete-{int(time.time())}"
        run_dir.rename(archive)
        print("Archived incomplete prior run to:", archive)
        run_dir = OUTPUT_DIR / RUN_NAME

    checkpoints = sorted(run_dir.glob("checkpoint-????????.pt")) if run_dir.exists() else []
    RESUME_FROM = checkpoints[-1] if (AUTO_RESUME and checkpoints) else None

    # On resume, restore the exact training config that is part of checkpoint identity.
    if RESUME_FROM is not None:
        cfg_path = run_dir / "training_config.json"
        assert cfg_path.is_file()
        TRAIN_CONFIG = json.loads(cfg_path.read_text(encoding="utf-8"))
        print("Restored training config from existing run.")

    training.seed_all(SEED, TRAIN_CONFIG.get("strict_determinism", False))
    core_model = EncoderDecoderLegalLM(MODEL_CONFIG).to("cuda:0")
    precision = training.precision_for(torch.device("cuda:0"), TRAIN_CONFIG["precision"])
    train_model = training.prepare_training_model(core_model, precision, device_ids=(0, 1))

    if RESUME_FROM is None and TRAIN_CONFIG.get("auto_microbatch", False):
        TRAIN_CONFIG = training.autotune_microbatch(train_model, TRAIN_CONFIG, torch.device("cuda:0"))

    plan = training.plan(MODEL_CONFIG, TRAIN_CONFIG)
    print("Resume:", RESUME_FROM)
    print("Target steps:", plan["steps"])
    print("Global physical batch:", TRAIN_CONFIG["physical_batch_size"])
    print("Gradient accumulation:", plan["gradient_accumulation_steps"])
    print("Planned raw tokens:", plan["planned_raw_tokens"])

    # If the newest checkpoint is already final, do not launch another training loop.
    if RESUME_FROM is not None:
        saved_state = torch.load(RESUME_FROM, map_location="cpu", weights_only=False)["state"]
        if saved_state["step"] >= plan["steps"]:
            print("Training already complete at step", saved_state["step"])
            FINAL_CHECKPOINT = Path(RESUME_FROM)
        else:
            result = training.train(
                train_model, train_data, val_data, TRAIN_CONFIG, OUTPUT_DIR,
                TOKENIZER_PATH, RUN_NAME, resume_from=RESUME_FROM,
                stop_after_step=SESSION_STOP_STEP,
            )
            FINAL_CHECKPOINT = Path(result["checkpoint"])
    else:
        result = training.train(
            train_model, train_data, val_data, TRAIN_CONFIG, OUTPUT_DIR,
            TOKENIZER_PATH, RUN_NAME, resume_from=None,
            stop_after_step=SESSION_STOP_STEP,
        )
        FINAL_CHECKPOINT = Path(result["checkpoint"])

    print("Checkpoint:", FINAL_CHECKPOINT)
else:
    print("Production training disabled. Set RUN_FULL_TRAINING=True after preflight passes.")

{'fit_probe_global_batch': 4, 'grad_accum': 8, 'peak_allocated_gib': [4.7, 4.14]}
Resume: None
Target steps: 762
Global physical batch: 4
Gradient accumulation: 8
Planned raw tokens: 49938432
{'step': 25, 'raw_tokens': 1638400, 'target_tokens': 409600, 'loss': 7.364101409912109, 'learning_rate': 0.00019994325455700542, 'gradient_norm': 1.4343043565750122, 'elapsed': 117.79650490300003, 'raw_tokens_per_second': 13908.731853709469}
{'step': 50, 'raw_tokens': 3276800, 'target_tokens': 819200, 'loss': 6.733487129211426, 'learning_rate': 0.00019903590262394466, 'gradient_norm': 0.5566169023513794, 'elapsed': 243.06495405500004, 'raw_tokens_per_second': 13481.170137174671}
{'step': 75, 'raw_tokens': 4915200, 'target_tokens': 1228800, 'loss': 6.617199897766113, 'learning_rate': 0.0001970318361885619, 'gradient_norm': 0.8173446655273438, 'elapsed': 367.58842032100006, 'raw_tokens_per_second': 13371.476706768279}
{'step': 100, 'raw_tokens': 6553600, 'target_tokens': 1638400, 'loss': 6.073251247

## Progress / ETA


In [17]:
run_dir = OUTPUT_DIR / RUN_NAME
log_path = run_dir / "training.jsonl"
if log_path.is_file():
    rows = [json.loads(x) for x in log_path.read_text(encoding="utf-8").splitlines() if x.strip()]
    latest = rows[-1]
    print(json.dumps(latest, indent=2))
    cfg = json.loads((run_dir / "training_config.json").read_text(encoding="utf-8"))
    p = training.plan(MODEL_CONFIG, cfg)
    speed = latest.get("raw_tokens_per_second", 0.0)
    if speed > 0:
        remaining = max(0, p["planned_raw_tokens"] - latest["raw_tokens"])
        print("Approx remaining hours:", round(remaining / speed / 3600, 2))
else:
    print("No training log yet.")

{
  "step": 762,
  "raw_tokens": 49938432,
  "target_tokens": 12484608,
  "loss": 4.775010108947754,
  "learning_rate": 8.867301043435916e-10,
  "gradient_norm": 0.47575607895851135,
  "elapsed": 3888.4643705910003,
  "raw_tokens_per_second": 12842.713020001249,
  "validation_loss": 4.870158906842842,
  "validation_perplexity": 130.34162742739804,
  "validation_targets": 249856,
  "validation_raw_tokens": 999424
}
Approx remaining hours: 0.0


## Fast cached generation helpers


In [18]:
def _word_spans(text):
    return list(re.finditer(r"\S+", text))


def _apply_repetition_penalty(logits, generated, penalty=1.10, window=128):
    if penalty <= 1.0 or not generated:
        return logits
    out = logits.clone()
    ids = list(set(generated[-window:]))
    if ids:
        idx = torch.tensor(ids, dtype=torch.long, device=out.device)
        vals = out[idx]
        out[idx] = torch.where(vals < 0, vals * penalty, vals / penalty)
    return out


def _blocked_ngram_tokens(generated, n=4):
    if n <= 1 or len(generated) < n - 1:
        return set()
    prefix = tuple(generated[-(n - 1):])
    blocked = set()
    for i in range(len(generated) - n + 1):
        if tuple(generated[i:i+n-1]) == prefix:
            blocked.add(generated[i+n-1])
    return blocked


def _sample(logits, tokenizer, generated, rng, temperature=0.1, top_p=0.95,
            repetition_penalty=1.10, no_repeat_ngram=4):
    logits = logits.float().clone()
    logits = _apply_repetition_penalty(logits, generated, repetition_penalty, 128)
    for special in (tokenizer.pad_token_id, tokenizer.unk_token_id,
                    tokenizer.bos_token_id, tokenizer.eos_token_id):
        if special is not None:
            logits[special] = float("-inf")
    before_ngram = logits.clone()
    for tid in _blocked_ngram_tokens(generated, no_repeat_ngram):
        logits[tid] = float("-inf")
    if not torch.isfinite(logits).any():
        logits = before_ngram
    if temperature == 0:
        return int(logits.argmax().item())
    sorted_logits, order = torch.sort(logits / temperature, descending=True)
    probs = sorted_logits.softmax(-1)
    remove = probs.cumsum(-1) - probs > top_p
    sorted_logits[remove] = float("-inf")
    pos = torch.multinomial(sorted_logits.softmax(-1), 1, generator=rng)
    return int(order[pos].item())


@torch.inference_mode()
def generate_exact_words(model, tokenizer, prompt, words=128, seed=42,
                         temperature=0.1, top_p=0.95,
                         repetition_penalty=1.10, no_repeat_ngram=4,
                         precision="auto"):
    model.eval()
    device = next(model.parameters()).device
    source = tokenizer.encode(prompt, add_special_tokens=False, split_special_tokens=True)
    if not source:
        raise ValueError("Prompt tokenized to zero tokens")
    source = source[-int(model.cfg["source_length"]):]
    source_ids = torch.tensor([source], dtype=torch.long, device=device)

    if precision == "auto":
        precision = "bf16" if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else ("fp16" if device.type == "cuda" else "fp32")
    amp_ctx = contextlib.nullcontext() if precision == "fp32" else torch.autocast(
        device_type=device.type, dtype=torch.bfloat16 if precision == "bf16" else torch.float16
    )

    with amp_ctx:
        memory = model.encode(source_ids)
        cross_cache = [block.prepare_cross(memory) for block in model.decoder_blocks]

    rng = torch.Generator(device=device).manual_seed(seed)
    self_cache = None
    generated = []
    current = tokenizer.bos_token_id

    for position in range(int(model.cfg["target_length"])):
        token = torch.tensor([[current]], dtype=torch.long, device=device)
        amp_ctx = contextlib.nullcontext() if precision == "fp32" else torch.autocast(
            device_type=device.type, dtype=torch.bfloat16 if precision == "bf16" else torch.float16
        )
        with amp_ctx:
            logits, self_cache = model.generation_step(token, position, cross_cache, self_cache)
        next_id = _sample(
            logits[0, -1], tokenizer, generated, rng,
            temperature=temperature, top_p=top_p,
            repetition_penalty=repetition_penalty,
            no_repeat_ngram=no_repeat_ngram,
        )
        generated.append(next_id)
        current = next_id

        if len(generated) % 8 == 0 or position + 1 == int(model.cfg["target_length"]):
            text = tokenizer.decode(
                generated, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )
            spans = _word_spans(text)
            if len(spans) >= words:
                exact = text[:spans[words - 1].end()].strip()
                assert len(exact.split()) == words
                return exact

    text = tokenizer.decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    raise ValueError(
        f"Only {len(_word_spans(text))} words produced within {model.cfg['target_length']} decoder positions. "
        f"Last fragment={text[-300:]!r}"
    )


def newest_checkpoint():
    cps = sorted((OUTPUT_DIR / RUN_NAME).glob("checkpoint-????????.pt"))
    if not cps:
        raise FileNotFoundError("No Model B checkpoint found")
    return cps[-1]


def load_inference_model(checkpoint_path=None):
    checkpoint_path = Path(checkpoint_path) if checkpoint_path is not None else newest_checkpoint()
    saved = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    assert saved["identity"]["tokenizer"] == TOKENIZER_HASH
    model = EncoderDecoderLegalLM(saved["identity"]["model"]).to("cuda:0" if torch.cuda.is_available() else "cpu")
    model.load_state_dict(saved["model"])
    model.eval()
    return model, checkpoint_path

## Optional qualitative generation


In [19]:
RUN_QUALITATIVE =True
PROMPT = "The court considered the submissions of both parties and held that"

if RUN_QUALITATIVE:
    inference_model, cp = load_inference_model()
    print("Checkpoint:", cp)
    text = generate_exact_words(inference_model, tokenizer, PROMPT, words=128, seed=SEED)
    print(text)
    print("Words:", len(text.split()))
    del inference_model
    gc.collect()
    torch.cuda.empty_cache()

Checkpoint: /kaggle/working/cs781-phase1-encdec-50m/encdec_5e5d_640_50m_2gpu/checkpoint-00000762.pt
.



The writ, the applicant is that the applicant is no.

In case of the applicant is not be released on the applicant shall be released on bail.

The applicant shall be filed by the applicant shall not be released to the court concerned with the applicant shall have been made by the applicant is disposed of the applicant shall appear before the applicant shall also been filed by the petitioner and the applicant is directed to the applicant shall cooperate in the applicant shall make any of the applicant.

Learned counsel for the applicant shall, the applicant shall justify the applicant shall proceed in the applicant is dismissed.

It is not tamper with the applicant is finally disposed of the case of the case, the applicant has
Words: 128


## Generation and submission


In [20]:
def _test_records(obj):
    if isinstance(obj, dict):
        for key, value in obj.items():
            text = value.get("text", value.get("prompt")) if isinstance(value, dict) else value
            if isinstance(text, str):
                yield str(key), text
        return
    if isinstance(obj, list):
        for i, value in enumerate(obj):
            assert isinstance(value, dict)
            rid = value.get("id", value.get("ID", i))
            text = value.get("text", value.get("prompt"))
            assert isinstance(text, str)
            yield str(rid), text
        return
    raise TypeError("Unsupported test.json structure")


if RUN_SUBMISSION:
    inference_model, checkpoint_path = load_inference_model(FINAL_CHECKPOINT)
    print("Submission checkpoint:", checkpoint_path)

    test_path = Path(hf_hub_download(
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        filename=TEST_FILENAME,
        revision=HF_DATASET_REVISION,
        token=HF_TOKEN,
    ))
    records = list(_test_records(json.loads(test_path.read_text(encoding="utf-8"))))
    assert records and len({rid for rid, _ in records}) == len(records)

    progress_path = OUTPUT_DIR / "encdec_submission_progress.json"
    predictions = {}
    if progress_path.is_file():
        old = json.loads(progress_path.read_text(encoding="utf-8"))
        predictions = {
            str(rid): value for rid, value in old.items()
            if isinstance(value, dict) and isinstance(value.get("text"), str)
            and len(value["text"].split()) == 128
        }

    valid_ids = {rid for rid, _ in records}
    predictions = {rid: value for rid, value in predictions.items() if rid in valid_ids}
    print("Already generated:", len(predictions), "/", len(records))

    for n, (rid, prompt) in enumerate(records, 1):
        if rid in predictions:
            print(f"[{n}/{len(records)}] id={rid} SKIPPED")
            continue

        last_error = None
        # Same decoding hyperparameters; retries change only the RNG seed.
        for attempt in range(3):
            try:
                continuation = generate_exact_words(
                    inference_model, tokenizer, prompt, words=128,
                    seed=SEED + n + attempt * 100003,
                    temperature=0.1, top_p=0.95,
                    repetition_penalty=1.10, no_repeat_ngram=4,
                )
                break
            except Exception as exc:
                last_error = exc
        else:
            raise RuntimeError(f"Generation failed for id={rid}: {last_error}")

        assert len(continuation.split()) == 128
        predictions[rid] = {"text": continuation}
        tmp = progress_path.with_suffix(".tmp")
        tmp.write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8")
        tmp.replace(progress_path)
        print(f"[{n}/{len(records)}] id={rid} DONE")

    assert set(predictions) == valid_ids
    ordered = {rid: predictions[rid] for rid, _ in records}
    assert all(len(v["text"].split()) == 128 for v in ordered.values())

    prediction_path = OUTPUT_DIR / "prediction.json"
    prediction_path.write_text(json.dumps(ordered, ensure_ascii=False, indent=2), encoding="utf-8")
    zip_path = OUTPUT_DIR / "submission.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(prediction_path, arcname="prediction.json")

    with zipfile.ZipFile(zip_path, "r") as zf:
        assert zf.namelist() == ["prediction.json"]
        inside = json.loads(zf.read("prediction.json").decode("utf-8"))
    assert set(inside) == valid_ids
    assert all(len(v["text"].split()) == 128 for v in inside.values())

    print("SUBMISSION COMPLETE")
    print("prediction.json:", prediction_path)
    print("submission.zip:", zip_path)

Submission checkpoint: /kaggle/working/cs781-phase1-encdec-50m/encdec_5e5d_640_50m_2gpu/checkpoint-00000762.pt


test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

Already generated: 0 / 1000
[1/1000] id=1 DONE
[2/1000] id=2 DONE
[3/1000] id=3 DONE
[4/1000] id=4 DONE
[5/1000] id=5 DONE
[6/1000] id=6 DONE
[7/1000] id=7 DONE
[8/1000] id=8 DONE
[9/1000] id=9 DONE
[10/1000] id=10 DONE
[11/1000] id=11 DONE
[12/1000] id=12 DONE
[13/1000] id=13 DONE
[14/1000] id=14 DONE
[15/1000] id=15 DONE
[16/1000] id=16 DONE
[17/1000] id=17 DONE
[18/1000] id=18 DONE
[19/1000] id=19 DONE
[20/1000] id=20 DONE
[21/1000] id=21 DONE
[22/1000] id=22 DONE
[23/1000] id=23 DONE
[24/1000] id=24 DONE
[25/1000] id=25 DONE
[26/1000] id=26 DONE
[27/1000] id=27 DONE
[28/1000] id=28 DONE
[29/1000] id=29 DONE
[30/1000] id=30 DONE
[31/1000] id=31 DONE
[32/1000] id=32 DONE
[33/1000] id=33 DONE
[34/1000] id=34 DONE
[35/1000] id=35 DONE
[36/1000] id=36 DONE
[37/1000] id=37 DONE
[38/1000] id=38 DONE
[39/1000] id=39 DONE
[40/1000] id=40 DONE
[41/1000] id=41 DONE
[42/1000] id=42 DONE
[43/1000] id=43 DONE
[44/1000] id=44 DONE
[45/1000] id=45 DONE
[46/1000] id=46 DONE
[47/1000] id=47 DONE
[48

## Optional checkpoint backup


In [21]:
MODEL_BACKUP_REPO = "I-230482/cs781-encdec-50m"  # e.g. "username/cs781-encdec-50m"

if MODEL_BACKUP_REPO:
    run_dir = OUTPUT_DIR / RUN_NAME
    assert run_dir.is_dir()
    api = HfApi(token=HF_TOKEN)
    api.create_repo(MODEL_BACKUP_REPO, repo_type="model", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(run_dir), repo_id=MODEL_BACKUP_REPO, repo_type="model")
    print("Backup complete:", MODEL_BACKUP_REPO)

Backup complete: I-230482/cs781-encdec-50m
